In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel




In [10]:
df = pd.read_csv("C:\\Users\\meswa\\OneDrive\\Desktop\\new file\\six_cle.csv")

In [11]:
X = df.drop("Price", axis=1)
y = df["Price"]

In [23]:
X.head()

,Company,TypeName,Ram,OpSys,Weight,TouchScreen,IPS,PPI,CPU_name,HDD,SSD,Gpu brand
0,1,4,8,0,1.37,0,1,226.983005,2,0,128,1
1,1,4,8,0,1.34,0,0,127.677940,2,0,0,1
2,7,3,8,1,1.86,0,0,141.211998,2,0,256,1
3,1,4,16,0,1.83,0,1,220.534624,3,0,512,0
4,1,4,8,0,1.37,0,1,226.983005,2,0,256,1


In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [25]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor




In [26]:


models = {
    "LinearRegression": {
        "model": LinearRegression(),
        "params": {}
    },
    
    "RandomForest": {
        "model": RandomForestRegressor(random_state=42),
        "params": {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, None]
        }
    },

    "knn": {
        "model": KNeighborsRegressor(),
        "params": {
            'n_neighbors': [3, 5, 7, 9, 11],
            'weights': ['uniform', 'distance'],
            'metric': ['euclidean', 'manhattan']
        }
    },
    
    "DecisionTree": {
        "model": DecisionTreeRegressor(),
        "params": {
            'max_depth': [5, 10, 20],
            'min_samples_split': [2, 5, 10]
        }
    },

   
    
    "SVR": {
        "model": SVR(),
        "params": {
            'C': [0.1, 1, 10],
            'kernel': ['linear', 'rbf']
        }
    }
}


best_models = {}


In [27]:
for name, config in models.items():
    print(f"\n Tuning {name}...")

    if config["params"]:  
        search = RandomizedSearchCV(
            config["model"],
            config["params"],
            n_iter=5,
            cv=3,
            random_state=42,
            n_jobs=-1
        )
        search.fit(X_train, y_train)

        best_model = search.best_estimator_
        print("Best Params:", search.best_params_)
    
    else:  
        best_model = config["model"]
        best_model.fit(X_train, y_train)
        
    y_pred = best_model.predict(X_test)
    score = r2_score(y_test, y_pred)
    score_mae = mean_absolute_error(y_test, y_pred)
    score_mse = mean_squared_error(y_test, y_pred)

    print("R2 Score:", score)
    print("mean_absolute_error:", score_mae)
    print("mean_squared_error:", score_mse)

    best_models[name] = {
        "model": best_model,
        "score": score
    }

best_algo = max(best_models, key=lambda x: best_models[x]["score"])

print("\n Best_algo:", best_algo)
print("Best Score:", best_models[best_algo]["score"])


 Tuning LinearRegression...
R2 Score: 0.715622342677224
mean_absolute_error: 15670.103710143267
mean_squared_error: 520128643.0630553

 Tuning RandomForest...
Best Params: {'n_estimators': 200, 'max_depth': 10}
R2 Score: 0.7961618746618194
mean_absolute_error: 11580.515303334343
mean_squared_error: 372821298.7433365

 Tuning knn...
Best Params: {'weights': 'distance', 'n_neighbors': 9, 'metric': 'manhattan'}
R2 Score: 0.6719414310106865
mean_absolute_error: 13685.544345742324
mean_squared_error: 600021323.5456359

 Tuning DecisionTree...
Best Params: {'min_samples_split': 10, 'max_depth': 20}
R2 Score: 0.7600088878755824
mean_absolute_error: 12917.670403678161
mean_squared_error: 438945354.11685276

 Tuning SVR...
Best Params: {'kernel': 'linear', 'C': 10}
R2 Score: 0.6713083498347874
mean_absolute_error: 15697.685912875773
mean_squared_error: 601179233.263541

 Best_algo: RandomForest
Best Score: 0.7961618746618194


In [28]:
import pickle


In [29]:
with open("best_algo.pkl", "wb") as f:
    pickle.dump(best_algo, f)